# 03 - Modelado Avanzado de ML con técnicas de ensemble 

En este notebook se evoluciona la fase de modelado aplicando técnicas de ensamble (*Ensemble Learning*) como **Gradient Boosting**, **XGBoost** y **LightGBM**. 

El objetivo es intentar mejorar las métricas obtenidas por el modelo baseline (Regresión Lineal, $R^2 \approx 0.8449$) mediante algoritmos basados en árboles de decisión secuenciales que corrigen los errores de sus predecesores.

In [19]:
import time
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Métricas
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Modelos de Ensemble
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor


sns.set_theme(style="whitegrid")

## Carga del dataset
Para garantizar que la comparación con el modelo baseline sea justa y reproducible, utilizaremos exactamente el mismo valor de random_state y reduciremos la precisión de los datos a float32 para optimizar la memoria RAM.

In [11]:
# Rutas habituales del proyecto
possible_paths = [
    Path("../data/raw/train.csv"),
    Path("data/raw/train.csv"),
    Path("train.csv"),
]

DATA_PATH = next((path for path in possible_paths if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError("No se encontró el archivo train.csv en las rutas especificadas.")

print(f"Ruta utilizada para entrenamiento avanzado: {DATA_PATH.resolve()}")
df = pd.read_csv(DATA_PATH)

# Optimización de tipos de datos
cols_float = df.select_dtypes(include="float64").columns
df[cols_float] = df[cols_float].astype("float32")

# Separación de variables
target = "FloodProbability"
feature_cols = [col for col in df.columns if col not in ["id", target]]

X = df[feature_cols]
y = df[target]

# División idéntica al notebook de Baseline (80% train / 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Muestra controlada de 100,000 registros para agilizar el entrenamiento
X_train_sample, _, y_train_sample, _ = train_test_split(X_train, y_train, train_size=100000, random_state=42)

print(f"Muestra de entrenamiento: {X_train_sample.shape}")
print(f"Conjunto de test: {X_test.shape}")

Ruta utilizada para entrenamiento avanzado: C:\Users\Personal\Proyecto-4--Grupo-1\data\raw\train.csv
Muestra de entrenamiento: (100000, 20)
Conjunto de test: (223592, 20)


## Definición de la Función de Evaluación


Reutilizamos la lógica del proyecto para calcular el RMSE, MAE, $R^2$ y el porcentaje de overfitting tanto en entrenamiento como en validación.

In [12]:
def evaluate_model(name, model, X_train_eval, y_train_eval, X_test_eval, y_test_eval):
    train_pred = model.predict(X_train_eval)
    test_pred = model.predict(X_test_eval)

    # Cálculo de métricas usando funciones importadas
    train_rmse = np.sqrt(mean_squared_error(y_train_eval, train_pred))
    train_mae = mean_absolute_error(y_train_eval, train_pred)
    train_r2 = r2_score(y_train_eval, train_pred)

    test_rmse = np.sqrt(mean_squared_error(y_test_eval, test_pred))
    test_mae = mean_absolute_error(y_test_eval, test_pred)
    test_r2 = r2_score(y_test_eval, test_pred)

    overfitting_r2_pct = abs(train_r2 - test_r2) / abs(train_r2) * 100 if train_r2 != 0 else np.nan

    return {
        "modelo": name,
        "RMSE_train": train_rmse,
        "MAE_train": train_mae,
        "R2_train": train_r2,
        "RMSE_test": test_rmse,
        "MAE_test": test_mae,
        "R2_test": test_r2,
        "overfitting_R2_%": overfitting_r2_pct,
        "cumple_overfitting": overfitting_r2_pct < 5,
        "train_pred": train_pred,
        "test_pred": test_pred,
    }

## 3. Evaluación del Modelo Baseline Anterior
Cargamos el archivo `.joblib` generado en el notebook anterior para evaluar sus métricas en esta misma ejecución y usarlo como punto de partida.

In [14]:
# Buscar el modelo baseline guardado
baseline_path = Path("../models/flood_baseline_model.joblib")
if not baseline_path.exists():
    baseline_path = Path("models/flood_baseline_model.joblib")

if baseline_path.exists():
    print(f"Cargando baseline desde: {baseline_path.resolve()}")
    lr_baseline = joblib.load(baseline_path)
    res_lr = evaluate_model("Linear Regression (Baseline)", lr_baseline, X_train_sample, y_train_sample, X_test, y_test)
    print(f"R2 del Baseline cargado: {res_lr['R2_test']:.4f}")
else:
    print("No se encontró el archivo del modelo baseline. Ejecuta el Notebook 02 para generar el modelo baseline antes de continuar.")

Cargando baseline desde: C:\Users\Personal\Proyecto-4--Grupo-1\models\flood_baseline_model.joblib
R2 del Baseline cargado: 0.8449


## 4. Entrenamiento de Modelos Avanzados (Boosting Ensembles)

A diferencia de los árboles tradicionales, los algoritmos de *Boosting* entrenan árboles de forma secuencial. Cada nuevo árbol se enfoca en minimizar los residuos (errores) cometidos por los árboles anteriores.

In [20]:
# Diccionario para almacenar los resultados completos de las predicciones
predictions = {}
models = {}

# Si el baseline se cargó correctamente, lo añadimos
if 'res_lr' in locals():
    models["Linear Regression (Baseline)"] = lr_baseline
    predictions["Linear Regression (Baseline)"] = res_lr["test_pred"]

# --- 1. Gradient Boosting Regressor ---
print("Entrenando Gradient Boosting de Scikit-Learn...")
gbr = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
inicio = time.time()
gbr.fit(X_train_sample, y_train_sample)
print(f"Tiempo: {time.time() - inicio:.1f} segundos.")

res_gbr = evaluate_model("Gradient Boosting", gbr, X_train_sample, y_train_sample, X_test, y_test)
models["Gradient Boosting"] = gbr
predictions["Gradient Boosting"] = res_gbr["test_pred"]


# --- 2. XGBoost Regressor ---
print("\nEntrenando XGBoost...")
xgb = XGBRegressor(n_estimators=150, learning_rate=0.05, max_depth=6, n_jobs=-1, random_state=42)
inicio = time.time()
xgb.fit(X_train_sample, y_train_sample)
print(f"Tiempo: {time.time() - inicio:.1f} segundos.")

res_xgb = evaluate_model("XGBoost", xgb, X_train_sample, y_train_sample, X_test, y_test)
models["XGBoost"] = xgb
predictions["XGBoost"] = res_xgb["test_pred"]


# --- 3. LightGBM Regressor ---
print("\nEntrenando LightGBM...")
lgb = LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=6, n_jobs=-1, random_state=42, verbose=-1)
inicio = time.time()
lgb.fit(X_train_sample, y_train_sample)
print(f"Tiempo: {time.time() - inicio:.1f} segundos.")

res_lgb = evaluate_model("LightGBM", lgb, X_train_sample, y_train_sample, X_test, y_test)
models["LightGBM"] = lgb
predictions["LightGBM"] = res_lgb["test_pred"]

# --- 4. Random Forest Regressor ---
print("\nEntrenando Random Forest...")
# n_jobs=-1 utiliza todos los núcleos de tu procesador para ir más rápido
# max_depth=800000 limita la profundidad para evitar que el modelo pese gigabytes en memoria
rf = RandomForestRegressor(n_estimators=100, max_depth=12, n_jobs=-1, random_state=42)
inicio = time.time()
rf.fit(X_train_sample, y_train_sample)
print(f"Tiempo: {time.time() - inicio:.1f} segundos.")

res_rf = evaluate_model("Random Forest", rf, X_train_sample, y_train_sample, X_test, y_test)
models["Random Forest"] = rf
predictions["Random Forest"] = res_rf["test_pred"]



Entrenando Gradient Boosting de Scikit-Learn...
Tiempo: 34.6 segundos.

Entrenando XGBoost...
Tiempo: 0.8 segundos.

Entrenando LightGBM...
Tiempo: 1.0 segundos.

Entrenando Random Forest...
Tiempo: 12.5 segundos.


# Tabla Comparativa de Resultados
Unificamos los resultados de todos los modelos entrenados para determinar cuál es el algoritmo definitivo que ofrece menor error (RMSE) y mayor capacidad de generalización ( R2 ).



In [21]:
# Construimos la lista filtrando los vectores de predicciones para que el dataframe sea legible
all_results = [res_gbr, res_xgb, res_lgb]
if 'res_lr' in locals():
    all_results.append(res_lr)

results_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ["train_pred", "test_pred"]} for r in all_results])

# Ordenamos de menor a mayor error en test
results_df = results_df.sort_values("RMSE_test", ascending=True)
results_df


,modelo,RMSE_train,MAE_train,R2_train,RMSE_test,MAE_test,R2_test,overfitting_R2_%,cumple_overfitting
3,Linear Regression (Baseline),0.020072,0.015816,0.845510,0.020081,0.015803,0.844858,0.077131,True
2,LightGBM,0.023991,0.019721,0.779300,0.025797,0.021187,0.743979,4.532441,True
0,Gradient Boosting,0.025236,0.020729,0.755786,0.026905,0.022080,0.721508,4.535475,True
1,XGBoost,0.025367,0.020888,0.753258,0.027791,0.022838,0.702863,6.690256,False


# Selección y Exportación del Modelo Ganador
Detectamos automáticamente el modelo con mejor desempeño en el conjunto de test y lo guardamos en la carpeta models/, sustituyendo o complementando el archivo anterior para su uso en producción (Streamlit).

In [22]:
best_row = results_df.iloc[0]
best_model_name = best_row["modelo"]

best_model = models[best_model_name]
best_pred = predictions[best_model_name]

print(f"El modelo definitivo es: {best_model_name}")

# Crear directorio si no existe
models_dir = Path("../models") if Path("../models").exists() else Path("models")
models_dir.mkdir(parents=True, exist_ok=True)

final_model_path = models_dir / "flood_final_ensemble_model.joblib"
joblib.dump(best_model, final_model_path)

print(f"Modelo final exportado con éxito en: {final_model_path.resolve()}")

El modelo definitivo es: Linear Regression (Baseline)
Modelo final exportado con éxito en: C:\Users\Personal\Proyecto-4--Grupo-1\models\flood_final_ensemble_model.joblib


# Conclusión: 
Evaluación de Modelos y Técnicas de EnsembleEn esta fase avanzada del proyecto, se ha evaluado el rendimiento de algoritmos basados en Ensemble Learning frente al modelo Baseline de Regresión Lineal ($R^2 \approx 0.8449$) utilizando una muestra controlada de 100,000 registros para optimizar el cómputo en local. Los modelos de ensamble implementados abarcaron tanto arquitecturas de Boosting Secuencial (Gradient Boosting Regressor, XGBoost y LightGBM) como de Bagging en Paralelo (Random Forest Regressor).Tras consolidar la tabla comparativa de métricas en el conjunto de test ($223,592$ registros), se determinó que el modelo definitivo y con mejor desempeño general es la Regresión Lineal (Baseline).

Justificación Técnica: ¿Por qué la Regresión Lineal es el Modelo Óptimo?
Para entender por qué un modelo clásico supera a algoritmos avanzados de Machine Learning en este proyecto, debemos analizar tres factores fundamentales:

1. La Evidencia del Análisis Exploratorio de Datos (EDA)
Durante la fase de exploración, al calcular la variable risk_score_sum (la suma directa de todos los factores de riesgo de inundación) y graficarla frente a nuestra variable objetivo FloodProbability, la distribución de los datos reveló una diagonal perfecta con dispersión constante. Esta simetría visual es la prueba reina de que el dataset (al ser de origen sintético) fue construido mediante una combinación lineal aditiva. En un escenario así, la Regresión Lineal no necesita aproximar; encuentra la ecuación matemática exacta que conecta las variables.

2. La Limitación Geométrica de los Árboles (El "Efecto Escalera")
Los modelos basados en árboles de decisión (Random Forest, XGBoost, LightGBM y Gradient Boosting) predicen fragmentando el espacio mediante umbrales ortogonales (condiciones binarias de "si/no").

El problema de la diagonal: Un árbol no tiene la capacidad física de trazar una línea diagonal continua. Para imitar una rampa o una línea recta ascendente, se ve obligado a construir miles de pequeños escalones horizontales y verticales.




D